In [21]:
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, accuracy_score

def set_seed(seed=97798760):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # make cudnn deterministic
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(507)

# T1
t1_train = pd.read_csv("../multi_omics/T1_train_selected.csv")
t1_test = pd.read_csv("../multi_omics/T1_test_selected.csv")

# T2
t2_train = pd.read_csv("../multi_omics/T2_train_selected.csv")
t2_test = pd.read_csv("../multi_omics/T2_test_selected.csv")

# ADC
adc_train = pd.read_csv("../multi_omics/ADC_train_selected.csv")
adc_test = pd.read_csv("../multi_omics/ADC_test_selected.csv")

def split_X_y(df):
    X = df.drop(columns=["label"])
    y = df["label"]
    return X, y

X_t1_train, y_t1_train = split_X_y(t1_train)
X_t1_test, y_t1_test = split_X_y(t1_test)

X_t2_train, y_t2_train = split_X_y(t2_train)
X_t2_test, y_t2_test = split_X_y(t2_test)

X_adc_train, y_adc_train = split_X_y(adc_train)
X_adc_test, y_adc_test = split_X_y(adc_test)

class MultiModalDataset(Dataset):
    def __init__(self, X_t1, X_t2, X_adc, y):
        self.X_t1 = torch.tensor(X_t1.values, dtype=torch.float32)
        self.X_t2 = torch.tensor(X_t2.values, dtype=torch.float32)
        self.X_adc = torch.tensor(X_adc.values, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_t1[idx],
            self.X_t2[idx],
            self.X_adc[idx],
            self.y[idx]
        )


train_dataset = MultiModalDataset(
    X_t1_train, X_t2_train, X_adc_train, y_t1_train
)

test_dataset = MultiModalDataset(
    X_t1_test, X_t2_test, X_adc_test, y_t1_test
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# Single Modality Neural Network

In [22]:
class SingleModalityNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=8, clf_hidden_dim=8, dropout=0.4):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, clf_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(clf_hidden_dim, 1)
        )

    def forward(self, x):
        h = self.encoder(x)
        logit = self.classifier(h).squeeze(1)
        return logit

def eval_single_from_multimodal(model, loader, device, modality="t1"):
    model.eval()

    all_y, all_prob, all_pred = [], [], []

    with torch.no_grad():
        for x_t1, x_t2, x_adc, y in loader:
            x_t1, x_t2, x_adc, y = (
                x_t1.to(device),
                x_t2.to(device),
                x_adc.to(device),
                y.to(device)
            )

            if modality == "t1":
                x = x_t1
            elif modality == "t2":
                x = x_t2
            elif modality == "adc":
                x = x_adc

            logits = model(x)
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()

            all_y.extend(y.cpu().numpy())
            all_prob.extend(probs.cpu().numpy())
            all_pred.extend(preds.cpu().numpy())

    all_y = np.array(all_y)
    all_prob = np.array(all_prob)
    all_pred = np.array(all_pred)

    acc = accuracy_score(all_y, all_pred)
    auc = roc_auc_score(all_y, all_prob) if len(np.unique(all_y)) > 1 else np.nan

    return acc, auc

def train_single_from_multimodal(
    model,
    train_loader,
    test_loader,
    device,
    modality="t1",
    epochs=20,
    lr=1e-3,
    weight_decay=1e-3
):
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for x_t1, x_t2, x_adc, y in train_loader:
            x_t1, x_t2, x_adc, y = (
                x_t1.to(device),
                x_t2.to(device),
                x_adc.to(device),
                y.to(device)
            )

            # pick modality
            if modality == "t1":
                x = x_t1
            elif modality == "t2":
                x = x_t2
            elif modality == "adc":
                x = x_adc
            else:
                raise ValueError("modality must be 't1', 't2', or 'adc'")

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * y.size(0)

        total_loss /= len(train_loader.dataset)

        train_acc, train_auc = eval_single_from_multimodal(model, train_loader, device, modality)
        test_acc, test_auc = eval_single_from_multimodal(model, test_loader, device, modality)

        if epoch == 0 or (epoch+1) % 10 == 0 :
            print(
                f"Epoch {epoch+1:03d} | Loss {total_loss:.4f} | "
                f"Train AUC {train_auc:.4f} | Train ACC {train_acc:.4f} |"
                f"Test AUC {test_auc:.4f} | Test ACC {test_acc:.4f}"
            )

    return (train_acc, train_auc, test_acc, test_auc)

In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

t1_model = SingleModalityNet(X_t1_train.shape[1])

train_single_from_multimodal(
    t1_model,
    train_loader,
    test_loader,
    device,
    lr=1e-4,
    weight_decay=1e-4,
    epochs=200,
    modality="t1"
)

Epoch 001 | Loss 0.6555 | Train AUC 0.6499 | Train ACC 0.7757 |Test AUC 0.6706 | Test ACC 0.8030
Epoch 010 | Loss 0.6469 | Train AUC 0.7872 | Train ACC 0.8099 |Test AUC 0.6993 | Test ACC 0.8333
Epoch 020 | Loss 0.6292 | Train AUC 0.8104 | Train ACC 0.8707 |Test AUC 0.7451 | Test ACC 0.8636
Epoch 030 | Loss 0.6094 | Train AUC 0.7971 | Train ACC 0.8897 |Test AUC 0.7752 | Test ACC 0.8636
Epoch 040 | Loss 0.5962 | Train AUC 0.8004 | Train ACC 0.8935 |Test AUC 0.8013 | Test ACC 0.8788
Epoch 050 | Loss 0.5709 | Train AUC 0.8024 | Train ACC 0.8935 |Test AUC 0.8131 | Test ACC 0.8788
Epoch 060 | Loss 0.5515 | Train AUC 0.8021 | Train ACC 0.8935 |Test AUC 0.8209 | Test ACC 0.8788
Epoch 070 | Loss 0.5283 | Train AUC 0.8044 | Train ACC 0.8973 |Test AUC 0.8366 | Test ACC 0.8788
Epoch 080 | Loss 0.5121 | Train AUC 0.8075 | Train ACC 0.9011 |Test AUC 0.8484 | Test ACC 0.8788
Epoch 090 | Loss 0.4968 | Train AUC 0.8084 | Train ACC 0.8973 |Test AUC 0.8510 | Test ACC 0.8788
Epoch 100 | Loss 0.4731 | Trai

(0.8935361216730038,
 0.8512277323062108,
 0.8787878787878788,
 0.8758169934640523)

In [24]:
t2_model = SingleModalityNet(X_t2_train.shape[1])

train_single_from_multimodal(
    t2_model,
    train_loader,
    test_loader,
    device,
    lr=1e-4,
    weight_decay=1e-4,
    epochs=200,
    modality="t2"
)

Epoch 001 | Loss 0.7064 | Train AUC 0.5673 | Train ACC 0.2586 |Test AUC 0.5778 | Test ACC 0.2727
Epoch 010 | Loss 0.6848 | Train AUC 0.7193 | Train ACC 0.6046 |Test AUC 0.5608 | Test ACC 0.4545
Epoch 020 | Loss 0.6639 | Train AUC 0.7666 | Train ACC 0.7985 |Test AUC 0.6327 | Test ACC 0.7121
Epoch 030 | Loss 0.6402 | Train AUC 0.8142 | Train ACC 0.8783 |Test AUC 0.7438 | Test ACC 0.8182
Epoch 040 | Loss 0.6148 | Train AUC 0.8630 | Train ACC 0.8859 |Test AUC 0.8235 | Test ACC 0.8182
Epoch 050 | Loss 0.6089 | Train AUC 0.8664 | Train ACC 0.8859 |Test AUC 0.8458 | Test ACC 0.8182
Epoch 060 | Loss 0.5701 | Train AUC 0.8659 | Train ACC 0.8859 |Test AUC 0.8444 | Test ACC 0.8333
Epoch 070 | Loss 0.5435 | Train AUC 0.8632 | Train ACC 0.8859 |Test AUC 0.8418 | Test ACC 0.8333
Epoch 080 | Loss 0.5222 | Train AUC 0.8610 | Train ACC 0.8859 |Test AUC 0.8405 | Test ACC 0.8333
Epoch 090 | Loss 0.5076 | Train AUC 0.8611 | Train ACC 0.8859 |Test AUC 0.8366 | Test ACC 0.8333
Epoch 100 | Loss 0.4705 | Trai

(0.8859315589353612,
 0.8567645642753972,
 0.8333333333333334,
 0.8222222222222222)

In [25]:
adc_model = SingleModalityNet(X_adc_train.shape[1])

train_single_from_multimodal(
    adc_model,
    train_loader,
    test_loader,
    device,
    lr=1e-4,
    weight_decay=1e-4,
    epochs=200,
    modality="adc"
)

Epoch 001 | Loss 0.7052 | Train AUC 0.8179 | Train ACC 0.4753 |Test AUC 0.6706 | Test ACC 0.5000
Epoch 010 | Loss 0.6813 | Train AUC 0.8433 | Train ACC 0.5856 |Test AUC 0.7255 | Test ACC 0.5303
Epoch 020 | Loss 0.6638 | Train AUC 0.8646 | Train ACC 0.8555 |Test AUC 0.7673 | Test ACC 0.7879
Epoch 030 | Loss 0.6653 | Train AUC 0.8772 | Train ACC 0.8555 |Test AUC 0.7948 | Test ACC 0.8030
Epoch 040 | Loss 0.6373 | Train AUC 0.8836 | Train ACC 0.8479 |Test AUC 0.8026 | Test ACC 0.8182
Epoch 050 | Loss 0.6291 | Train AUC 0.8879 | Train ACC 0.8479 |Test AUC 0.7935 | Test ACC 0.8333
Epoch 060 | Loss 0.6293 | Train AUC 0.8927 | Train ACC 0.8479 |Test AUC 0.7830 | Test ACC 0.8333
Epoch 070 | Loss 0.6014 | Train AUC 0.8924 | Train ACC 0.8517 |Test AUC 0.7869 | Test ACC 0.8333
Epoch 080 | Loss 0.6011 | Train AUC 0.8973 | Train ACC 0.8517 |Test AUC 0.7830 | Test ACC 0.8485
Epoch 090 | Loss 0.5804 | Train AUC 0.8978 | Train ACC 0.8555 |Test AUC 0.7863 | Test ACC 0.8485
Epoch 100 | Loss 0.5653 | Trai

(0.8745247148288974,
 0.8957631198844488,
 0.8333333333333334,
 0.7934640522875817)

# Non-gated Fusion Neural Network

In [26]:
class ModalityEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=8, dropout=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)
    
class EncodedConcatFusionNet(nn.Module):
    def __init__(
        self,
        t1_dim,
        t2_dim,
        adc_dim,
        hidden_dim=16,
        clf_hidden_dim=16,
        dropout=0.2
    ):
        super().__init__()

        self.t1_encoder = ModalityEncoder(t1_dim, hidden_dim, dropout)
        self.t2_encoder = ModalityEncoder(t2_dim, hidden_dim, dropout)
        self.adc_encoder = ModalityEncoder(adc_dim, hidden_dim, dropout)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 3, clf_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(clf_hidden_dim, 1)
        )

    def forward(self, x_t1, x_t2, x_adc):
        h_t1 = self.t1_encoder(x_t1)
        h_t2 = self.t2_encoder(x_t2)
        h_adc = self.adc_encoder(x_adc)

        h_fused = torch.cat([h_t1, h_t2, h_adc], dim=1)
        logit = self.classifier(h_fused).squeeze(1)

        return logit

In [27]:
def evaluate_concat_model(model, loader, device):
    model.eval()

    all_y, all_prob, all_pred = [], [], []

    with torch.no_grad():
        for x_t1, x_t2, x_adc, y in loader:
            x_t1 = x_t1.to(device)
            x_t2 = x_t2.to(device)
            x_adc = x_adc.to(device)
            y = y.to(device)

            logits = model(x_t1, x_t2, x_adc)
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()

            all_y.extend(y.cpu().numpy())
            all_prob.extend(probs.cpu().numpy())
            all_pred.extend(preds.cpu().numpy())

    all_y = np.array(all_y)
    all_prob = np.array(all_prob)
    all_pred = np.array(all_pred)

    acc = accuracy_score(all_y, all_pred)
    auc = roc_auc_score(all_y, all_prob) if len(np.unique(all_y)) > 1 else np.nan

    return {
        "acc": acc,
        "auc": auc,
        "y_true": all_y,
        "y_prob": all_prob,
        "y_pred": all_pred
    }

def train_concat_model(model, train_loader, test_loader, device, epochs=100, lr=1e-3, weight_decay=1e-4):
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for x_t1, x_t2, x_adc, y in train_loader:
            x_t1 = x_t1.to(device)
            x_t2 = x_t2.to(device)
            x_adc = x_adc.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(x_t1, x_t2, x_adc)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * y.size(0)

        total_loss /= len(train_loader.dataset)

        train_result = evaluate_concat_model(model, train_loader, device)
        test_result = evaluate_concat_model(model, test_loader, device)

        if epoch == 0 or (epoch+1) % 10 == 0:
            print(
                f"Epoch {epoch+1:03d} | "
                f"Loss {total_loss:.4f} | "
                f"Train AUC {train_result['auc']:.4f} | "
                f"Train ACC {train_result['acc']:.4f} | "
                f"Test AUC {test_result['auc']:.4f} | "
                f"Test ACC {test_result['acc']:.4f}"
            )

    return model


In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
concat_model = EncodedConcatFusionNet(
    t1_dim=X_t1_train.shape[1],
    t2_dim=X_t2_train.shape[1],
    adc_dim=X_adc_train.shape[1],
    hidden_dim=8,
    clf_hidden_dim=8,
    dropout=0.4
)

concat_model = train_concat_model(
    concat_model,
    train_loader,
    test_loader,
    device,
    epochs=200,
    lr=1e-4,
    weight_decay=1e-4
)

Epoch 001 | Loss 0.6975 | Train AUC 0.3998 | Train ACC 0.5361 | Test AUC 0.1974 | Test ACC 0.4697
Epoch 010 | Loss 0.6613 | Train AUC 0.5665 | Train ACC 0.7529 | Test AUC 0.3046 | Test ACC 0.6818
Epoch 020 | Loss 0.6467 | Train AUC 0.6721 | Train ACC 0.8213 | Test AUC 0.5542 | Test ACC 0.7879
Epoch 030 | Loss 0.6264 | Train AUC 0.6967 | Train ACC 0.8593 | Test AUC 0.6052 | Test ACC 0.8030
Epoch 040 | Loss 0.6073 | Train AUC 0.7179 | Train ACC 0.8783 | Test AUC 0.6771 | Test ACC 0.8333
Epoch 050 | Loss 0.5912 | Train AUC 0.7287 | Train ACC 0.8783 | Test AUC 0.6889 | Test ACC 0.8636
Epoch 060 | Loss 0.5618 | Train AUC 0.7367 | Train ACC 0.8783 | Test AUC 0.7059 | Test ACC 0.8636
Epoch 070 | Loss 0.5508 | Train AUC 0.7468 | Train ACC 0.8859 | Test AUC 0.7268 | Test ACC 0.8788
Epoch 080 | Loss 0.5275 | Train AUC 0.7635 | Train ACC 0.8859 | Test AUC 0.7673 | Test ACC 0.8788
Epoch 090 | Loss 0.5090 | Train AUC 0.7910 | Train ACC 0.8897 | Test AUC 0.8288 | Test ACC 0.8788
Epoch 100 | Loss 0.4

# Global Weight Fusion Neural Network

In [29]:
class GlobalWeightFusionNet(nn.Module):
    def __init__(
        self,
        t1_dim,
        t2_dim,
        adc_dim,
        hidden_dim=8,
        clf_hidden_dim=8,
        dropout=0.4
    ):
        super().__init__()

        self.t1_encoder = ModalityEncoder(t1_dim, hidden_dim, dropout)
        self.t2_encoder = ModalityEncoder(t2_dim, hidden_dim, dropout)
        self.adc_encoder = ModalityEncoder(adc_dim, hidden_dim, dropout)

        # trainable global fusion logits -> softmax -> global weights
        self.global_gate_logits = nn.Parameter(torch.zeros(3))

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, clf_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(clf_hidden_dim, 1)
        )

    def forward(self, x_t1, x_t2, x_adc):
        h_t1 = self.t1_encoder(x_t1)
        h_t2 = self.t2_encoder(x_t2)
        h_adc = self.adc_encoder(x_adc)

        global_weights = torch.softmax(self.global_gate_logits, dim=0)  # shape: (3,)

        h_fused = (
            global_weights[0] * h_t1 +
            global_weights[1] * h_t2 +
            global_weights[2] * h_adc
        )

        logit = self.classifier(h_fused).squeeze(1)

        return logit, global_weights

In [30]:
def evaluate_global_weight_model(model, loader, device):
    model.eval()

    all_y, all_prob, all_pred = [], [], []
    learned_weights = None

    with torch.no_grad():
        for x_t1, x_t2, x_adc, y in loader:
            x_t1 = x_t1.to(device)
            x_t2 = x_t2.to(device)
            x_adc = x_adc.to(device)
            y = y.to(device)

            logits, global_weights = model(x_t1, x_t2, x_adc)
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()

            all_y.extend(y.cpu().numpy())
            all_prob.extend(probs.cpu().numpy())
            all_pred.extend(preds.cpu().numpy())

            learned_weights = global_weights.detach().cpu().numpy()

    all_y = np.array(all_y)
    all_prob = np.array(all_prob)
    all_pred = np.array(all_pred)

    acc = accuracy_score(all_y, all_pred)
    auc = roc_auc_score(all_y, all_prob) if len(np.unique(all_y)) > 1 else np.nan

    return {
        "acc": acc,
        "auc": auc,
        "global_weights": learned_weights,
        "y_true": all_y,
        "y_prob": all_prob,
        "y_pred": all_pred
    }

def train_global_weight_model(model, train_loader, test_loader, device, epochs=100, lr=1e-3, weight_decay=1e-4):
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for x_t1, x_t2, x_adc, y in train_loader:
            x_t1 = x_t1.to(device)
            x_t2 = x_t2.to(device)
            x_adc = x_adc.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits, global_weights = model(x_t1, x_t2, x_adc)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * y.size(0)

        total_loss /= len(train_loader.dataset)

        train_result = evaluate_global_weight_model(model, train_loader, device)
        test_result = evaluate_global_weight_model(model, test_loader, device)

        if epoch == 0 or (epoch+1) % 10 == 0:
            print(
                f"Epoch {epoch+1:03d} | "
                f"Loss {total_loss:.4f} | "
                f"Train AUC {train_result['auc']:.4f} | "
                f"Train ACC {train_result['acc']:.4f} | "
                # check confusion matrix if needed
                # f"Train Confusion Matrix:\n{pd.crosstab(train_result['y_true'], train_result['y_pred'], rownames=['True'], colnames=['Pred'])}\n"
                f"Test AUC {test_result['auc']:.4f} | "
                f"Test ACC {test_result['acc']:.4f}"
                #f"Test Confusion Matrix:\n{pd.crosstab(test_result['y_true'], test_result['y_pred'], rownames=['True'], colnames=['Pred'])}\n"
            )
            print("Global weights [T1, T2, ADC]:", np.round(test_result["global_weights"], 4))

    return model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

global_model = GlobalWeightFusionNet(
    t1_dim=X_t1_train.shape[1],
    t2_dim=X_t2_train.shape[1],
    adc_dim=X_adc_train.shape[1],
    hidden_dim=8,
    clf_hidden_dim=8,
    dropout=0.4
)

global_model = train_global_weight_model(
    global_model,
    train_loader,
    test_loader,
    device,
    epochs=500,
    lr=1e-4,
    weight_decay=1e-4
)

Epoch 001 | Loss 0.7054 | Train AUC 0.2035 | Train ACC 0.5095 | Test AUC 0.1477 | Test ACC 0.4091
Global weights [T1, T2, ADC]: [0.3335 0.3334 0.3331]
Epoch 010 | Loss 0.6939 | Train AUC 0.3391 | Train ACC 0.6426 | Test AUC 0.3333 | Test ACC 0.6667
Global weights [T1, T2, ADC]: [0.3346 0.3337 0.3317]
Epoch 020 | Loss 0.6694 | Train AUC 0.6212 | Train ACC 0.7795 | Test AUC 0.6614 | Test ACC 0.7879
Global weights [T1, T2, ADC]: [0.3358 0.3344 0.3298]
Epoch 030 | Loss 0.6509 | Train AUC 0.7354 | Train ACC 0.8023 | Test AUC 0.7660 | Test ACC 0.8030
Global weights [T1, T2, ADC]: [0.3373 0.3344 0.3283]
Epoch 040 | Loss 0.6447 | Train AUC 0.7765 | Train ACC 0.8365 | Test AUC 0.8144 | Test ACC 0.8182
Global weights [T1, T2, ADC]: [0.3389 0.3346 0.3265]
Epoch 050 | Loss 0.6357 | Train AUC 0.8045 | Train ACC 0.8593 | Test AUC 0.8471 | Test ACC 0.8333
Global weights [T1, T2, ADC]: [0.3409 0.3351 0.324 ]
Epoch 060 | Loss 0.6189 | Train AUC 0.8231 | Train ACC 0.8745 | Test AUC 0.8824 | Test ACC 0.8

# Gated Adaptive Fusion Neural Network

In [32]:
class GatedAdaptiveFusionNet(nn.Module):
    def __init__(
        self,
        t1_dim,
        t2_dim,
        adc_dim,
        hidden_dim=16,
        gate_hidden_dim=8,
        clf_hidden_dim=8,
        dropout=0.4
    ):
        super().__init__()

        # modality-specific encoders
        self.t1_encoder = ModalityEncoder(t1_dim, hidden_dim, dropout)
        self.t2_encoder = ModalityEncoder(t2_dim, hidden_dim, dropout)
        self.adc_encoder = ModalityEncoder(adc_dim, hidden_dim, dropout)

        # gate network: outputs 3 modality weights
        self.gate_net = nn.Sequential(
            nn.Linear(hidden_dim * 3, gate_hidden_dim),
            nn.ReLU(),
            nn.Linear(gate_hidden_dim, 3)
        )

        # classifier on fused representation
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, clf_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(clf_hidden_dim, 1)
        )

    def forward(self, x_t1, x_t2, x_adc):
        # encode each modality
        h_t1 = self.t1_encoder(x_t1)    # (B, hidden_dim)
        h_t2 = self.t2_encoder(x_t2)
        h_adc = self.adc_encoder(x_adc)

        # concatenate latent representations
        h_cat = torch.cat([h_t1, h_t2, h_adc], dim=1)   # (B, 3 * hidden_dim)

        # gating weights
        gate_logits = self.gate_net(h_cat)              # (B, 3)
        gates = torch.softmax(gate_logits, dim=1)       # each row sums to 1

        # weighted fusion
        h_fused = (
            gates[:, 0:1] * h_t1 +
            gates[:, 1:2] * h_t2 +
            gates[:, 2:3] * h_adc
        )

        # final logit
        logit = self.classifier(h_fused).squeeze(1)     # (B,)

        return logit, gates

In [33]:
def evaluate_model(model, data_loader, device):
    model.eval()

    all_y = []
    all_prob = []
    all_pred = []
    all_gates = []

    with torch.no_grad():
        for x_t1, x_t2, x_adc, y in data_loader:
            x_t1 = x_t1.to(device)
            x_t2 = x_t2.to(device)
            x_adc = x_adc.to(device)
            y = y.to(device)

            logits, gates = model(x_t1, x_t2, x_adc)
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()

            all_y.extend(y.cpu().numpy())
            all_prob.extend(probs.cpu().numpy())
            all_pred.extend(preds.cpu().numpy())
            all_gates.append(gates.cpu().numpy())

    all_y = np.array(all_y)
    all_prob = np.array(all_prob)
    all_pred = np.array(all_pred)
    all_gates = np.vstack(all_gates)

    acc = accuracy_score(all_y, all_pred)

    # AUC requires both classes present
    if len(np.unique(all_y)) == 2:
        auc = roc_auc_score(all_y, all_prob)
    else:
        auc = np.nan

    mean_gates = all_gates.mean(axis=0)
    var_gates = all_gates.var(axis=0)

    return {
        "acc": acc,
        "auc": auc,
        "mean_gates": mean_gates,
        "var_gates": var_gates,
        "y_true": all_y,
        "y_prob": all_prob,
        "y_pred": all_pred,
        "all_gates": all_gates
    }


def train_model(
    model,
    train_loader,
    test_loader,
    device,
    lr=1e-3,
    weight_decay=1e-4,
    num_epochs=100
):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0

        for x_t1, x_t2, x_adc, y in train_loader:
            x_t1 = x_t1.to(device)
            x_t2 = x_t2.to(device)
            x_adc = x_adc.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            logits, gates = model(x_t1, x_t2, x_adc)
            loss = criterion(logits, y)

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * y.size(0)

        epoch_loss /= len(train_loader.dataset)

        train_result = evaluate_model(model, train_loader, device)
        test_result = evaluate_model(model, test_loader, device)

        if epoch == 0 or (epoch+1) % 10 == 0:
            print(
            f"Epoch {epoch+1:03d} | "
            f"Loss: {epoch_loss:.4f} | "
            f"Train AUC: {train_result['auc']:.4f} | "
            f"Train ACC: {train_result["acc"]:.4f} | "
            # check confusion matrix if
            # f"Train Confusion Matrix:\n{pd.crosstab(train_result['y_true'], train_result['y_pred'], rownames=['True'], colnames=['Pred'])}\n"
            f"Test AUC: {test_result['auc']:.4f} | "
            f"Test ACC: {test_result['acc']:.4f}"
            # f"Test Confusion Matrix:\n{pd.crosstab(test_result['y_true'], test_result['y_pred'], rownames=['True'], colnames=['Pred'])}\n"
            )
            print(
                f"Mean train gates [T1, T2, ADC]: {np.round(train_result['mean_gates'], 4)}",
                f"Std train gates [T1, T2, ADC]: {np.round(train_result['var_gates'], 4)}"
            )
    return model

In [51]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GatedAdaptiveFusionNet(
    t1_dim=X_t1_train.shape[1],
    t2_dim=X_t2_train.shape[1],
    adc_dim=X_adc_train.shape[1],
    hidden_dim=8,
    gate_hidden_dim=8,
    clf_hidden_dim=8,
    dropout=0.4
).to(device)

model = train_model(
    model,
    train_loader,
    test_loader,
    device,
    lr=1e-4,
    weight_decay=1e-4,
    num_epochs=500
)



Epoch 001 | Loss: 0.6582 | Train AUC: 0.3675 | Train ACC: 0.7643 | Test AUC: 0.3673 | Test ACC: 0.7727
Mean train gates [T1, T2, ADC]: [0.2767 0.3532 0.37  ] Std train gates [T1, T2, ADC]: [0.0007 0.0002 0.0002]
Epoch 010 | Loss: 0.6432 | Train AUC: 0.6709 | Train ACC: 0.7643 | Test AUC: 0.6575 | Test ACC: 0.7727
Mean train gates [T1, T2, ADC]: [0.2748 0.3536 0.3717] Std train gates [T1, T2, ADC]: [6.e-04 1.e-04 2.e-04]
Epoch 020 | Loss: 0.6280 | Train AUC: 0.7733 | Train ACC: 0.7643 | Test AUC: 0.7895 | Test ACC: 0.7727
Mean train gates [T1, T2, ADC]: [0.2773 0.3546 0.3681] Std train gates [T1, T2, ADC]: [0.0007 0.0002 0.0003]
Epoch 030 | Loss: 0.6142 | Train AUC: 0.8252 | Train ACC: 0.7757 | Test AUC: 0.8523 | Test ACC: 0.7727
Mean train gates [T1, T2, ADC]: [0.2855 0.3532 0.3614] Std train gates [T1, T2, ADC]: [0.0013 0.0003 0.0005]
Epoch 040 | Loss: 0.5971 | Train AUC: 0.8556 | Train ACC: 0.8099 | Test AUC: 0.8850 | Test ACC: 0.7879
Mean train gates [T1, T2, ADC]: [0.293  0.3541 0.